<a href="https://colab.research.google.com/github/sara78606/UniLex/blob/main/UniLex_CS_Full_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UniLex: CS Dataset

## Step 1 : Install dependencies

In [5]:
!pip install -q transformers accelerate torch pandas tqdm

## Step 2 : Upload both raw source files
Click **Choose Files** and select BOTH at once:
- `wikipedia_cs_glossary_raw.csv`
- `technical_concept_simplifier_final.csv`

In [6]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

wiki_df = pd.read_csv('wikipedia_cs_glossary_raw.csv')
tech_df = pd.read_csv('technical_concept_simplifier_final.csv')
cleaned = pd.read_csv('cs_dataset_merged_clean.csv')

print(f"Wikipedia glossary: {len(wiki_df)} rows")
print(f"Technical simplifier: {len(tech_df)} rows")
print(f"Cleaned: {len(cleaned)} rows")

Saving cs_dataset_merged_clean.csv to cs_dataset_merged_clean (3).csv
Saving technical_concept_simplifier_final.csv to technical_concept_simplifier_final (1).csv
Saving wikipedia_cs_glossary_raw.csv to wikipedia_cs_glossary_raw (2).csv
Wikipedia glossary: 159 rows
Technical simplifier: 999 rows
Cleaned: 1150 rows


## Step 3 : Preprocess Source 1: Wikipedia CS Glossary
Has `term` + `definition`. We standardize it into the unified schema.
The fields it's missing (`student_friendly_explanation`, `example`, `category`, `difficulty`, `related_terms`)
are left blank here later will be filled.


In [7]:
import re

def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = re.sub(r'\s+', ' ', s)          # collapse whitespace
    s = re.sub(r'\[\d+\]', '', s)        # remove leftover citation markers like [12]
    return s.strip()

wiki_clean = pd.DataFrame()
wiki_clean['term'] = wiki_df['term'].apply(clean_text).str.title()
wiki_clean['formal_definition'] = wiki_df['definition'].apply(clean_text)
wiki_clean['student_friendly_explanation'] = None
wiki_clean['example'] = None
wiki_clean['category'] = None
wiki_clean['difficulty'] = None
wiki_clean['related_terms'] = None
wiki_clean['source'] = 'wikipedia_glossary'

# drop empty/too-short rows
wiki_clean = wiki_clean[wiki_clean['term'].str.len() > 1]
wiki_clean = wiki_clean[wiki_clean['formal_definition'].str.len() > 10]
wiki_clean = wiki_clean.drop_duplicates(subset='term', keep='first').reset_index(drop=True)

print(f"Wikipedia source cleaned: {len(wiki_clean)} rows")
wiki_clean.head(3)

Wikipedia source cleaned: 159 rows


,term,formal_definition,student_friendly_explanation,example,category,difficulty,related_terms,source
0,Abstract Data Type,A mathematical model for data types in which a...,None,None,None,None,None,wikipedia_glossary
1,Abstract Method,One with only a signature and no implementatio...,None,None,None,None,None,wikipedia_glossary
2,Abstraction,"In software engineering and computer science, ...",None,None,None,None,None,wikipedia_glossary


## Step 4 : Preprocess Source 2: Technical Concept Simplifier
Has `instruction` (e.g. "Explain JVM Internals") + `response` (explanation blended with example).
We extract a clean `term`, split the response into `student_friendly_explanation` and `example`.
`formal_definition` is left blank it will be generated earlier.

In [8]:
def extract_term(instruction):
    s = str(instruction).strip()
    s = re.sub(r'^(explain|what is|what are|describe|define)\s+', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\?$', '', s).strip()
    return s.title()

EXAMPLE_MARKERS = [
    r'a real-world example', r'real-world example', r'for instance',
    r'think of it as', r'think of', r'imagine'
]
example_split_re = re.compile('(' + '|'.join(EXAMPLE_MARKERS) + ')', flags=re.IGNORECASE)

def split_response(response):
    text = clean_text(response)
    match = example_split_re.search(text)
    if match and match.start() > 20:
        return text[:match.start()].strip(), text[match.start():].strip()
    return text, ""

tech_clean = pd.DataFrame()
tech_clean['term'] = tech_df['instruction'].apply(extract_term)

splits = tech_df['response'].apply(split_response)
tech_clean['student_friendly_explanation'] = [s[0] for s in splits]
tech_clean['example'] = [s[1] for s in splits]
tech_clean['formal_definition'] = None
tech_clean['category'] = None
tech_clean['difficulty'] = None
tech_clean['related_terms'] = None
tech_clean['source'] = 'technical_concept_simplifier'

tech_clean = tech_clean[tech_clean['term'].str.len() > 1]
tech_clean = tech_clean[tech_clean['student_friendly_explanation'].str.len() > 10]
tech_clean = tech_clean.drop_duplicates(subset='term', keep='first').reset_index(drop=True)

print(f"Technical simplifier source cleaned: {len(tech_clean)} rows")
tech_clean.head(3)

Technical simplifier source cleaned: 995 rows


,term,student_friendly_explanation,example,formal_definition,category,difficulty,related_terms,source
0,Jvm Internals To A Beginner,The Java Virtual Machine or JVM is the engine ...,Think of it as a virtual computer that runs on...,None,None,None,None,technical_concept_simplifier
1,Java Garbage Collection And Why Do We Need It,Garbage Collection in Java is an automatic mem...,,None,None,None,None,technical_concept_simplifier
2,Can You Simplify Java Class Loaders,Java Class Loaders are part of the Java Virtua...,Think of it like a streaming service like Netf...,None,None,None,None,technical_concept_simplifier


## Step 5 : Merge both sources into one unified dataset

In [9]:
SCHEMA = ['term', 'formal_definition', 'student_friendly_explanation', 'example',
          'category', 'difficulty', 'related_terms', 'source']

merged = pd.concat([wiki_clean[SCHEMA], tech_clean[SCHEMA]], ignore_index=True)
print(f"Merged: {len(merged)} rows ({len(wiki_clean)} wikipedia + {len(tech_clean)} technical_simplifier)")
merged.head(3)

Merged: 1154 rows (159 wikipedia + 995 technical_simplifier)


,term,formal_definition,student_friendly_explanation,example,category,difficulty,related_terms,source
0,Abstract Data Type,A mathematical model for data types in which a...,None,None,None,None,None,wikipedia_glossary
1,Abstract Method,One with only a signature and no implementatio...,None,None,None,None,None,wikipedia_glossary
2,Abstraction,"In software engineering and computer science, ...",None,None,None,None,None,wikipedia_glossary


## Step 6 : Clean the merged dataset
Now that both sources share one schema, do a final cleanup pass:
- drop exact/near duplicate terms across BOTH sources (case-insensitive)
- strip stray whitespace again
- drop rows with no usable text at all
- reset the index

In [10]:
cleaned = merged.copy()

for col in ['term', 'formal_definition', 'student_friendly_explanation', 'example']:
    cleaned[col] = cleaned[col].apply(clean_text)

# dedupe on lowercased term, keep the Wikipedia version when both sources have it
# (Wikipedia definitions tend to be more precise/formal)
cleaned['_term_lower'] = cleaned['term'].str.lower()
cleaned = cleaned.sort_values(by='source', key=lambda s: s != 'wikipedia_glossary')
cleaned = cleaned.drop_duplicates(subset='_term_lower', keep='first')
cleaned = cleaned.drop(columns=['_term_lower'])

# a row needs at least SOME real content beyond just a term
has_content = (cleaned['formal_definition'].str.len() > 10) | (cleaned['student_friendly_explanation'].str.len() > 10)
cleaned = cleaned[has_content]

cleaned = cleaned.reset_index(drop=True)
print(f"Cleaned merged dataset: {len(cleaned)} rows")
cleaned['source'].value_counts()

Cleaned merged dataset: 1150 rows


,count
source,
technical_concept_simplifier,991
wikipedia_glossary,159


## Step 7 : Save the cleaned merged dataset


In [11]:
cleaned.to_csv('cs_dataset_merged_clean.csv', index=False)
files.download('cs_dataset_merged_clean.csv')
print("Saved and downloaded cs_dataset_merged_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloaded cs_dataset_merged_clean.csv


## Step 8 : Load Qwen2.5-3B-Instruct

In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)
print("Model loaded.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


## Step 9 : Define the labeling function
This is now schema-aware: it fills in whatever fields are missing per row
(Wikipedia rows get `student_friendly_explanation` + `example` generated;
Technical-simplifier rows get `formal_definition` generated),
and ALWAYS generates `category`, `difficulty`, and `related_terms` for every row.

In [14]:
import json

CATEGORIES = [
    "Programming Fundamentals", "Data Structures", "Algorithms", "Databases",
    "Operating Systems", "Computer Networks", "Cloud Computing",
    "Software Engineering", "Artificial Intelligence", "Cybersecurity"
]

def build_prompt(row):
    needs_formal_def = not row['formal_definition']
    needs_explanation = not row['student_friendly_explanation']
    needs_example = not row['example']

    known = f"Term: {row['term']}\n"
    if row['formal_definition']:
        known += f"Formal definition: {row['formal_definition']}\n"
    if row['student_friendly_explanation']:
        known += f"Student explanation: {row['student_friendly_explanation']}\n"
    if row['example']:
        known += f"Example: {row['example']}\n"

    fields_to_generate = ['"category"', '"difficulty"', '"related_terms"']
    if needs_formal_def:
        fields_to_generate.insert(0, '"formal_definition"')
    if needs_explanation:
        fields_to_generate.append('"student_friendly_explanation"')
    if needs_example:
        fields_to_generate.append('"example"')

    instructions = f"""You are labeling an entry for a Computer Science glossary dataset aimed at students.

{known}
Fill in ONLY the missing fields below. Keep any given content as-is (do not repeat it in your answer).

- category: pick the SINGLE best fit from exactly this list: {', '.join(CATEGORIES)}
- difficulty: one of beginner, intermediate, advanced
- related_terms: 3 real, related CS terms (not made up)
{'- formal_definition: one precise academic sentence defining the term' if needs_formal_def else ''}
{'- student_friendly_explanation: a simple, friendly 2-3 sentence explanation for a student' if needs_explanation else ''}
{'- example: one short real-world example illustrating the term' if needs_example else ''}

Respond with ONLY valid JSON containing exactly these keys: {', '.join(fields_to_generate)}
No other text."""
    return instructions

def generate_labels(row):
    prompt = build_prompt(row)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=300, temperature=0.3,
            do_sample=True, pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    match = re.search(r'\{.*\}', response, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {}

## Step 10 : Test on 3 rows first

In [15]:
for i in range(3):
    row = cleaned.iloc[i]
    result = generate_labels(row)
    print(f"Term: {row['term']} (source: {row['source']})")
    print(f"Generated: {result}\n")

Term: Abstract Data Type (source: wikipedia_glossary)
Generated: {'category': 'Programming Fundamentals', 'difficulty': 'intermediate', 'related_terms': ['Data Structure', 'Object-Oriented Programming', 'Software Design'], 'student_friendly_explanation': "An Abstract Data Type is like a blueprint for how data should behave, without worrying about how it's actually stored or implemented. It helps us think about what we can do with the data, not how to do it.", 'example': 'A stack, where you can push items onto it and pop them off, is an example of an Abstract Data Type.'}

Term: Machine Learning (source: wikipedia_glossary)
Generated: {'category': 'Artificial Intelligence', 'difficulty': 'Intermediate', 'related_terms': ['AI', 'Deep Learning', 'Neural Networks'], 'student_friendly_explanation': 'Machine Learning is like teaching a computer to learn from examples instead of following strict rules. It helps computers get better at tasks as they see more data.', 'example': 'Self-driving ca

## Step 11 : Run on the full merged, cleaned dataset
~1-2 sec/row on a free T4. Checkpoints save every 100 rows in case Colab disconnects.

In [16]:
from tqdm import tqdm

final = cleaned.copy()

for i in tqdm(range(len(final))):
    row = final.iloc[i]
    result = generate_labels(row)

    if not row['formal_definition'] and 'formal_definition' in result:
        final.at[i, 'formal_definition'] = result['formal_definition']
    if not row['student_friendly_explanation'] and 'student_friendly_explanation' in result:
        final.at[i, 'student_friendly_explanation'] = result['student_friendly_explanation']
    if not row['example'] and 'example' in result:
        final.at[i, 'example'] = result['example']

    final.at[i, 'category'] = result.get('category')
    final.at[i, 'difficulty'] = result.get('difficulty')
    related = result.get('related_terms', [])
    final.at[i, 'related_terms'] = '; '.join(related) if isinstance(related, list) else str(related)

    if (i + 1) % 100 == 0:
        final.to_csv('cs_dataset_checkpoint.csv', index=False)
        print(f"Checkpoint saved at row {i+1}")

  9%|▊         | 100/1150 [10:31<1:48:29,  6.20s/it]

Checkpoint saved at row 100


 17%|█▋        | 200/1150 [19:38<1:10:09,  4.43s/it]

Checkpoint saved at row 200


 26%|██▌       | 300/1150 [27:36<1:40:48,  7.12s/it]

...

 87%|████████▋ | 1000/1150 [1:25:46<14:17,  5.71s/it]

Checkpoint saved at row 1000


 96%|█████████▌| 1100/1150 [1:34:50<04:28,  5.37s/it]

Checkpoint saved at row 1100


100%|██████████| 1150/1150 [1:39:20<00:00,  5.18s/it]


## Step 12 : Save and download the final dataset

In [17]:
final.to_csv('cs_dataset_final.csv', index=False)
files.download('cs_dataset_final.csv')
print(f"Done. Final dataset: {len(final)} rows.")
final.head(5)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done. Final dataset: 1150 rows.


,term,formal_definition,student_friendly_explanation,example,category,difficulty,related_terms,source
0,Abstract Data Type,A mathematical model for data types in which a...,An abstract data type is like a blueprint for ...,"A stack, where you can push items onto it and ...",Programming Fundamentals,intermediate,Data Structure; Object-Oriented Programming; S...,wikipedia_glossary
1,Machine Learning,A subfield of artificial intelligence in which...,Machine Learning is like teaching a computer t...,Google's spam filter uses Machine Learning to ...,Artificial Intelligence,Intermediate,AI; Data Mining; Deep Learning,wikipedia_glossary
2,Memory Leak,A software bug where a program fails to releas...,A memory leak happens when a program doesn't g...,Imagine you have a toy box that gets smaller e...,Software Engineering,intermediate,Garbage Collection; Resource Management; Resou...,wikipedia_glossary
3,Metadata,"Data that describes other data, such as a file...",Metadata is like when you write down who wrote...,"When you open a PDF file, metadata might tell ...",Databases,beginner,Data; Primary Data; Secondary Data,wikipedia_glossary
4,Middleware,Software that acts as a bridge between differe...,Middleware is like a helper between different ...,A chat application uses middleware to connect ...,Software Engineering,intermediate,API; API Gateway; API Management,wikipedia_glossary
